In [ ]:
%pip install bert_score sentence_transformers osmnx trl unsloth

In [ ]:
import torch
from sentence_transformers import util

class OSMSemanticBridge:
    def __init__(self, tag_schema, st_model, threshold=0.80):
        """
        Initializes the bridge using a structured schema rather than raw graph crawling.
        
        Args:
            tag_schema (dict): The dictionary containing 'continuous' and 'discrete' OSM keys.
            st_model: The SentenceTransformer instance.
            threshold (float): Similarity floor for a 'synset' match.
        """
        self.st_model = st_model
        self.threshold = threshold
        self.schema = tag_schema
        
        # We focus on discrete categories as they represent the 'Synsets' for NLP mapping
        self.categories = list(tag_schema["discrete"].keys())
        
        # 1. Targeted Indexing: Build a semantic map for each category in the schema
        print(f"Bridge: Building Schema-Grounded Index for categories: {self.categories}")
        self.category_indices = {}
        
        for cat in self.categories:
            raw_values = tag_schema["discrete"][cat]
            # Clean values for better embedding (e.g., 'motorway_link' -> 'motorway link')
            clean_values = [str(v).lower().replace('_', ' ') for v in raw_values]
            
            # Pre-cache vectors for this specific category
            embeddings = self.st_model.encode(clean_values, convert_to_tensor=True)
            
            self.category_indices[cat] = {
                "original": raw_values,
                "clean": clean_values,
                "embeddings": embeddings
            }

    def get_osm_synsets(self, entity_text: str):
        """
        Maps a natural language entity to specific OSM keys and values.
        
        Returns:
            dict: { 'highway': ['motorway_link', 'trunk_link'], 'junction': ['roundabout'] }
        """
        entity_vec = self.st_model.encode(entity_text.lower(), convert_to_tensor=True)
        mapped_synsets = {}
        
        for cat, index in self.category_indices.items():
            # Calculate similarity against this category's possible values
            scores = util.cos_sim(entity_vec, index["embeddings"])[0]
            
            # Find all matches above the threshold
            match_indices = torch.where(scores > self.threshold)[0]
            
            if len(match_indices) > 0:
                mapped_synsets[cat] = [index["original"][i] for i in match_indices]
        
        return mapped_synsets

In [ ]:
import numpy as np
import torch
import networkx as nx
from typing import List, Dict
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score_fn
import nltk
from nltk.corpus import stopwords
import spacy
import re

class prompt_based_route_evaluator:
    def __init__(self, graph, prompts: List[str], routes: List[List[int]], bridge, device: str = "cpu"):
        self.graph = graph
        self.prompts = prompts
        self.routes = routes
        self.bridge = bridge
        self.device = device
        
        # 1. Critic Model: Independent from Actor's MiniLM
        self.st_evaluator = SentenceTransformer('sentence-transformers/multi-qa-mpnet-base-dot-v1', device=device)
        
        # 2. NLP Infrastructure
        self.nlp = spacy.load("en_core_web_sm")
            
        self.stop_words = set(stopwords.words('english'))
        
        # 3. Pre-cached Polarity Anchor
        self.avoid_concept = self.st_evaluator.encode(
            "avoid exclude shun shunned away from bypass miss without", 
            convert_to_tensor=True
        )

    def check_path_validity(self, route: List[int]) -> bool:
        """Verifies if the path is physically traversable in the graph."""
        if not route or len(route) < 2: return False
        return all(self.graph.has_edge(u, v) for u, v in zip(route[:-1], route[1:]))

    def deviation_penalty(self, route: List[int]) -> float:
        """Calculates length ratio between generated route and shortest physical path."""
        if not self.check_path_validity(route): return 10.0
        start, end = route[0], route[-1]
        try:
            # ideal_len uses pure physical length
            ideal_len = nx.shortest_path_length(self.graph, start, end, weight='length')
            # actual_len sums physical length of the generated route
            actual_len = 0
            for u, v in zip(route[:-1], route[1:]):
                edge_data = self.graph.get_edge_data(u, v)
                data = list(edge_data.values())[0] if isinstance(edge_data, dict) else edge_data
                actual_len += data.get('length', 1.0)
            
            return actual_len / ideal_len if ideal_len > 0 else 1.0
        except: 
            return 10.0

    def _get_path_tags_list(self, route: List[int]) -> List[str]:
        """Extracts unique discrete tags based on the bridge's dynamic schema."""
        tags = set()
        schema_keys = self.bridge.categories
        for u, v in zip(route[:-1], route[1:]):
            edge_data = self.graph.get_edge_data(u, v)
            if not edge_data: continue
            data = list(edge_data.values())[0] if isinstance(edge_data, dict) else edge_data
            
            for key in schema_keys:
                val = data.get(key)
                if val:
                    if isinstance(val, list):
                        tags.update([str(v).lower().replace('_', ' ') for v in val])
                    else:
                        tags.add(str(val).lower().replace('_', ' '))
        return list(tags)

    def _get_path_metadata_string(self, route: List[int]) -> str:
        """Flattens all relevant tags (schema + names) into a semantic sentence."""
        # 1. Start with schema-grounded tags
        all_tags = self._get_path_tags_list(route)
        
        # 2. Augment with names and descriptive tags for BERTScore global vibe
        for u, v in zip(route[:-1], route[1:]):
            edge_data = self.graph.get_edge_data(u, v)
            data = list(edge_data.values())[0] if isinstance(edge_data, dict) else edge_data
            for key in ['name', 'amenity', 'shop', 'leisure', 'landuse']:
                val = data.get(key)
                if val:
                    if isinstance(val, list):
                        all_tags.extend([str(v).lower().replace('_', ' ') for v in val])
                    else:
                        all_tags.append(str(val).lower().replace('_', ' '))
        
        return " ".join(list(set(all_tags))).lower()

    def semantic_alignment_bertscore(self) -> float:
        """Calculates global F1 alignment between prompt and path metadata."""
        if not self.routes: return 0.0
        candidates = [self._get_path_metadata_string(r) for r in self.routes]
        references = self.prompts
        
        _, _, f1 = bert_score_fn(candidates, references, lang="en", device=self.device, verbose=False)
        return f1.mean().item()

    def constraint_satisfaction(self, prompt: str, route: List[int]) -> float:
        """Fine-grained satisfaction using dependency logic and schema synsets."""
        doc = self.nlp(prompt.lower())
        path_tags = self._get_path_tags_list(route)
        path_metadata_str = self._get_path_metadata_string(route)
        if not path_metadata_str: return 0.0

        semantic_constraints = []
        for chunk in doc.noun_chunks:
            clean_text = " ".join(re.findall(r"[\w-]+", chunk.text))
            if all(w in self.stop_words for w in clean_text.split()):
                continue

            # Check for linguistic negation ('no', 'not')
            is_negated = any(t.dep_ == "neg" or t.lemma_ in ["no", "not"] for t in chunk)
            
            # Check for semantic negation in head verb (e.g., 'avoid')
            head_verb = chunk.root.head
            if head_verb.pos_ == "VERB":
                verb_emb = self.st_evaluator.encode(head_verb.lemma_, convert_to_tensor=True)
                if torch.cosine_similarity(verb_emb.unsqueeze(0), self.avoid_concept.unsqueeze(0)) > 0.6:
                    is_negated = True

            semantic_constraints.append({"text": clean_text, "negated": is_negated})

        if not semantic_constraints: return 1.0

        total_satisfaction = 0
        for c in semantic_constraints:
            # 1. Symbolic: Check synset mapping from the bridge
            synset_map = self.bridge.get_osm_synsets(c['text'])
            target_tags = [val for values in synset_map.values() for val in values]
            has_hard_match = any(tag in path_tags for tag in target_tags)
            
            # 2. Neural: Cosine similarity fallback
            c_emb = self.st_evaluator.encode(c['text'], convert_to_tensor=True)
            p_emb = self.st_evaluator.encode(path_metadata_str, convert_to_tensor=True)
            similarity = float(torch.cosine_similarity(c_emb.unsqueeze(0), p_emb.unsqueeze(0)).item())

            current_score = 1.0 if has_hard_match else similarity

            # 3. Polarity Logic
            if c['negated']:
                total_satisfaction += (1.0 - current_score)
            else:
                total_satisfaction += current_score

        return total_satisfaction / len(semantic_constraints)

    def evaluate_method(self) -> Dict:
        """Aggregates all metrics for the batch."""
        if not self.routes: return {"error": "No routes to evaluate"}

        # BERTScore is computed batch-wise for GPU efficiency
        avg_semantic_align = self.semantic_alignment_bertscore()
        
        return {
            "avg_path_validity": float(np.mean([self.check_path_validity(r) for r in self.routes])),
            "avg_deviation_penalty": float(np.mean([self.deviation_penalty(r) for r in self.routes])),
            "avg_constraint_satisfaction": float(np.mean([self.constraint_satisfaction(p, r) for p, r in zip(self.prompts, self.routes)])),
            "avg_semantic_alignment_f1": avg_semantic_align
        }

In [ ]:
import torch
import json
import pickle
import osmnx as ox
import networkx as nx
import re
from trl import GRPOConfig, GRPOTrainer
from unsloth import FastLanguageModel

# 1. Load your existing graph (Global Access)
with open("research/pioneer_valley_v2.pkl", "rb") as f:
    G = pickle.load(f)

# 2. Load Llama-3.2-3B via Unsloth (Memory Efficient)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=1024,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(model, r=32, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])

# 3. Define the GRPO Reward Function (Neuro-Symbolic Bridge)
def grpo_reward_func(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        # A. Parse the LLM's 12-dimensional vector output
        try:
            match = re.search(r"\[([0-9.,\s\-]+)\]", completion)
            vector = json.loads(match.group(0))
            if len(vector) != 12: raise ValueError
        except:
            rewards.append(-10.0) # Penalty for bad syntax
            continue
            
        # B. Run A* with the warped weights (The Symbolic Engine)
        # item metadata needs to be extracted from prompt context
        try:
            # Here: You'd parse nodes from the prompt (stored in a dictionary map)
            s_node, e_node = get_nodes_from_prompt(prompt) 
            route = run_warped_astar(G, s_node, e_node, vector)
            
            # C. Calculate Reward (Constraint Satisfaction vs. Distance)
            dist_score = calculate_route_distance(route)
            const_score = bridge.calculate_constraint_score(route, prompt) # Your Synset logic
            
            # Reward = Alpha * Constraints - Beta * Distance
            rewards.append((2.0 * const_score) - (1.0 * dist_score))
        except:
            rewards.append(-5.0)
    return rewards

# 4. GRPO Configuration
training_args = GRPOConfig(
    output_dir="grpo-pioneer-router",
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=500,
)

# 5. Trainer
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=grpo_reward_func,
    args=training_args,
    train_dataset=dataset_for_grpo, # List of prompt strings
)

trainer.train()


In [ ]:
import json
import pickle
import osmnx as ox
import networkx as nx
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from eval.evaluation import prompt_based_route_evaluator
from routing.NER import GlinerAStarRouter
from routing.synset import OSMSemanticBridge

# The Schema defines the technical 'Search Space' for the Bridge
tag_schema = {
    "continuous": {
        "maxspeed_imputed": {"min": 10, "max": 65, "unit": "mph"},
        "lanes": {"min": 1, "max": 6},
        "length": {"min": 2, "max": 6845}
    },
    "discrete": {
        "highway": ['residential', 'trunk', 'secondary', 'tertiary', 'primary', 'motorway_link', 
                     'unclassified', 'secondary_link', 'motorway', 'tertiary_link', 'primary_link', 'trunk_link'],
        "access": ["yes", "no"],
        "bridge": ["yes", "viaduct"],
        "junction": ["roundabout", "jughandle"],
        "ref": ['US 20', 'US 5', 'MA 9', 'MA 10', 'MA 66', 'MA 187', 'US 202', 'I 91', 'I 90', 'MA 116']
    }
}

if __name__ == "__main__":
    mode = 'astar'
    
    # 1. Load Data
    graph_path = "research/pioneer_valley_v2.pkl"
    with open(graph_path, "rb") as f:
        G = pickle.load(f)

    with open("research/synthetic_dataset.jsonl", "r") as f:
        data = [json.loads(line) for line in f]

    # 2. Initialize Models
    # Actor/Bridge model (Efficiency focused)
    st_model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # NEW: Initialize the Bridge with the Schema instead of raw Graph Crawling
    # This prevents the 'accidental' mapping of noisy graph strings
    bridge = OSMSemanticBridge(tag_schema, st_model, threshold=0.80)
    
    # Initialize the Router with high-contrast weights for better Dijkstra/A* differentiation
    # [Penalty Multiplier, Reward Multiplier]
    router = GlinerAStarRouter(G, [25.0, 0.01], bridge)
    
    gen_routes, gen_prompts = [], []

    print(f"\n--- Running Neurosymbolic Experiment (Mode: {mode}) ---")
    # Limiting batch for testing: 100:210
    for item in tqdm(data[100:200]):
        try:
            # Geocode to lat/lon with specific regional context
            s_pt = ox.geocoder.geocode(f"{item['start']}, MA, USA")
            e_pt = ox.geocoder.geocode(f"{item['end']}, MA, USA")
            
            s_node = ox.distance.nearest_nodes(G, X=s_pt[1], Y=s_pt[0])
            e_node = ox.distance.nearest_nodes(G, X=e_pt[1], Y=e_pt[0])
            
            if s_node == e_node: continue
            if not nx.has_path(G, s_node, e_node): continue

            # Routing execution using our GLiNER + Schema logic
            route = router.find_route(s_node, e_node, item['prompt'], algorithm=mode)
            
            if route:
                gen_routes.append(route)
                gen_prompts.append(item['prompt'])
                
        except Exception:
            continue

    # 3. Final Evaluation
    if gen_routes:
        print(f"\nSuccessfully generated {len(gen_routes)} routes. Evaluating...")
        # The Evaluator now uses spaCy and NLTK internally for the 'Better Way' logic
        evaluator = prompt_based_route_evaluator(G, gen_prompts, gen_routes, bridge)
        results = evaluator.evaluate_method()
        
        print("\n--- Final Consolidated Metrics ---")
        for metric, value in results.items():
            # Formatting the print to handle the new satisfaction scores
            print(f"{metric:30}: {value:.4f}")
            
        # Optional: Print a single prompt/route logic check
        print(f"\nSample Prompt: {gen_prompts[0]}")
    else:
        print("\n[!] No valid routes were generated. Check graph connectivity or geocoding.")